In [ ]:
# 1. Download Data
# Google Drive
!gdown --id '149paISvxCXDlr-720UEzseQA-y53rZxN' --output food-11.zip

# Dropbox
# !wget "https://www.dropbox.com/s/7yl5rra84ia0k8f/food-11.zip?dl=0" -O food-11.zip

# MEGA
# !wget https://megatools.megous.com/builds/megatools-1.10.3.tar.gz
# !tar -zxvf /content/megatools-1.10.3.tar.gz
# !sudo apt-get install libtool libglib2.0-dev gobject-introspection libgmp3-dev nettle-dev asciidoc glib-networking openssl libcurl4-openssl-dev libssl-dev
# %cd megatools-1.10.3/
# !./configure make
# !sudo make install
# %cd /content/
# !megadl 'https://mega.nz/file/FdlygByK#QQ5LP71MMjeXrXwoXM2qlygUsJ1D-6d5fMhj5gyi2Vc'

!unzip food-11.zip

In [ ]:
# 2. Import Packages
import torch
import torch.nn as nn
import torchvision.transforms as transforms

import numpy as np
from PIL import Image

from torch.utils.data import ConcatDataset, DataLoader, Subset, TensorDataset
from torchvision.datasets import DatasetFolder

from tqdm import tqdm

In [ ]:
# 3. Transforms

# not every augmentation is useful.
# Please think about what kind of augmentation is helpful for food recognition.
train_tfm = transforms.Compose(
    [
        transforms.Resize((128, 128)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        transforms.RandomResizedCrop(128, scale=(0.8, 1.0)),
        transforms.ToTensor(),
    ]
)

# We don't need augmentations in testing and validation.
# All we need here is to resize the PIL image and transform it into Tensor.
test_tfm = transforms.Compose(
    [
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
    ]
)

In [ ]:
# 4. Dataset and DataLoader
batch_size = 128

train_set = DatasetFolder(
    "food-11/training/labeled",
    loader=lambda x: Image.open(x),
    extensions="jpg",
    transform=train_tfm,
)
valid_set = DatasetFolder(
    "food-11/validation",
    loader=lambda x: Image.open(x),
    extensions="jpg",
    transform=test_tfm,
)
unlabeled_set = DatasetFolder(
    "food-11/training/unlabeled",
    loader=lambda x: Image.open(x),
    extensions="jpg",
    transform=train_tfm,
)
unlabeled_set_pseudo = DatasetFolder(
    "food-11/training/unlabeled",
    loader=lambda x: Image.open(x),
    extensions="jpg",
    transform=test_tfm,  # 使用乾淨資料預測未知標籤
)
test_set = DatasetFolder(
    "food-11/testing",
    loader=lambda x: Image.open(x),
    extensions="jpg",
    transform=test_tfm,
)

train_loader = DataLoader(
    train_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True
)
valid_loader = DataLoader(
    valid_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

In [ ]:
# 5. 建立模型
import torchvision.models as models
import torch.nn as nn

class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.resnet18(weights=None)
        in_features = self.model.fc.in_features
        self.model.fc = nn.Linear(in_features, 11)

    def forward(self, x):
        return self.model(x)

In [ ]:
# # 6. 半監督學習 (Pseudo-labeling)
# def get_pseudo_labels(dataset, model, threshold=0.85):
#     device = "cuda" if torch.cuda.is_available() else "cpu"
#     data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
#     model.eval()
#     softmax = nn.Softmax(dim=-1)

#     # 儲存符合條件的影像與偽標籤
#     pseudo_indices_list, pseudo_labels_list = [], []
#     sample_idx = 0

#     for batch in tqdm(data_loader, desc="Pseudo-labeling"):
#         imgs, _ = batch
#         with torch.no_grad():
#             logits = model(imgs.to(device))
#         probs = softmax(logits)
#         max_probs, pred_labels = torch.max(probs, dim=1)

#         # 過濾高信心樣本
#         mask = max_probs > threshold
#         if mask.any():
#             batch_size_actual = len(imgs)
#             batch_indices = torch.arange(sample_idx, sample_idx + batch_size_actual, device="cpu")[mask.cpu()]
#             pseudo_indices_list.append(batch_indices)
#             pseudo_labels_list.append(pred_labels[mask].cpu())

#         sample_idx += len(imgs)

#     model.train()

#     # 回傳符合高信心門檻的偽標籤資料集
#     if len(pseudo_indices_list) == 0:
#         print("No pseudo-labels generated (all below threshold).")
#         return [], torch.tensor([])

#     pseudo_indices = torch.cat(pseudo_indices_list, dim=0).tolist()
#     pseudo_labels = torch.cat(pseudo_labels_list, dim=0)
#     return pseudo_indices, pseudo_labels

In [ ]:
# get_pseudo_labels_v2（回傳索引 + 標籤，支援強增強）
def get_pseudo_labels_v2(dataset, model, threshold=0.85):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    model.eval()
    softmax = nn.Softmax(dim=-1)

    pseudo_indices_list, pseudo_labels_list = [], []
    sample_idx = 0

    for batch in tqdm(data_loader, desc="Pseudo-labeling"):
        imgs, _ = batch
        with torch.no_grad():
            logits = model(imgs.to(device))
        probs = softmax(logits)
        max_probs, pred_labels = torch.max(probs, dim=1)

        mask = max_probs > threshold
        if mask.any():
            batch_size_actual = len(imgs)
            batch_indices = torch.arange(sample_idx, sample_idx + batch_size_actual, device="cpu")[mask.cpu()]
            pseudo_indices_list.append(batch_indices)
            pseudo_labels_list.append(pred_labels[mask].detach().cpu())

        sample_idx += len(imgs)

    model.train()

    if len(pseudo_indices_list) == 0:
        print("No pseudo-labels generated (all below threshold).")
        return [], torch.tensor([])

    pseudo_indices = torch.cat(pseudo_indices_list, dim=0).tolist()
    pseudo_labels = torch.cat(pseudo_labels_list, dim=0)

    return pseudo_indices, pseudo_labels

In [ ]:
# 7. 訓練迴圈（Train + Valid）
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Classifier().to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80)

do_semi = True

class PseudoDataset(torch.utils.data.Dataset):
    def __init__(self, subset_images, labels):
        self.subset_images = subset_images
        self.labels = labels
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        # 這裡會觸發 unlabeled_set 的 train_tfm，使影像在訓練時得到隨機強增強
        img, _ = self.subset_images[idx]
        label = self.labels[idx]
        return img, label

for epoch in range(80):
    # 半監督：產生偽標籤。簡化後的 DataLoader 切換
    train_set_to_use = train_set
    if do_semi and epoch >= 5:
        pseudo_indices, pseudo_labels = get_pseudo_labels_v2(unlabeled_set_pseudo, model, threshold=0.85)
        if len(pseudo_indices) > 0:
            # 使用索引去抓取擁有「強增強（train_tfm）」的影像子集
            from torch.utils.data import Subset
            strong_pseudo_images_set = Subset(unlabeled_set, pseudo_indices)

            pseudo_set = PseudoDataset(strong_pseudo_images_set, pseudo_labels)
            train_set_to_use = ConcatDataset([train_set, pseudo_set])

    train_loader = DataLoader(
        train_set_to_use, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True
    )

    # === Training ===
    model.train()
    train_loss = 0.0
    train_accs = []
    for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}"):
        imgs, labels = batch
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        acc = (logits.argmax(dim=-1) == labels).float().mean()
        train_loss += loss.item()
        train_accs.append(acc)

    train_loss = train_loss / len(train_loader)
    train_acc = sum(train_accs) / len(train_accs)
    scheduler.step()

    # === Validation ===
    model.eval()
    valid_loss = 0.0
    valid_accs = []
    with torch.no_grad():
        for batch in valid_loader:
            imgs, labels = batch
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss = criterion(logits, labels)
            acc = (logits.argmax(dim=-1) == labels).float().mean()
            valid_loss += loss.item()
            valid_accs.append(acc)
    valid_loss = valid_loss / len(valid_loader)
    valid_acc = sum(valid_accs) / len(valid_accs)

    print(f"Epoch {epoch + 1:03d} | Train Acc: {train_acc:.4f} | Valid Acc: {valid_acc:.4f}")

In [ ]:
# 8. 測試推論
model.eval()
predictions = []


for batch in tqdm(test_loader):
    # A batch consists of image data and corresponding labels.
    # But here the variable "labels" is useless since we do not have the ground-truth.
    # If printing out the labels, you will find that it is always 0.
    # This is because the wrapper (DatasetFolder) returns images and labels for each batch,
    # so we have to create fake labels to make it work normally.
    imgs, labels = batch

    # We don't need gradient in testing, and we don't even have labels to compute loss.
    with torch.no_grad():
        logits = model(imgs.to(device))

    # Take the class with greatest logit as prediction and record it.
    predictions.extend(logits.argmax(dim=-1).cpu().numpy().tolist())

In [ ]:
# 9. 輸出 CSV 儲存
with open("predict.csv", "w") as f:
    f.write("Id,Category\n")

    # For the rest of the rows, each image id corresponds to a predicted class.
    for i, pred in enumerate(predictions):
        f.write(f"{i},{pred}\n")

In [ ]:
# 印出產生了多少偽標籤
print(f"Generated {len(pseudo_indices)} pseudo-labeled samples.")